In [675]:
import sys
import json

general = {'use_road_network': True, 'n_clusters_vononoi': 250, 'plot_ODs': False, 'transfer_penalty': 600}
footpaths = {'max_length': 1000, 'speed': 3.5, 'n_ntlegs': 5}
pathfinder_params = {'period': ''}

params = {'footpaths': footpaths, 'general': general, 'pathfinder params (headway)': pathfinder_params}

default = {'training_folder': '../../', 'scenario': 'clermont_v4', 'params': params}
manual, argv = (True, default) if 'ipykernel' in sys.argv[0] else (False, dict(default, **json.loads(sys.argv[1])))
print(argv)

{'training_folder': '../../', 'scenario': 'clermont_v4', 'params': {'footpaths': {'max_length': 1000, 'speed': 3.5, 'n_ntlegs': 5}, 'general': {'use_road_network': True, 'n_clusters_vononoi': 250, 'plot_ODs': False, 'transfer_penalty': 600}, 'pathfinder params (headway)': {'period': ''}}}


In [676]:
import os
import geopandas as gpd
import pandas as pd
import polars as pl
import numpy as np
import tqdm
import shapely
import math

sys.path.insert(0, r'../../../quetzal')  # Add path to quetzal
from shapely.geometry import Point, LineString
from shapely import union_all
from quetzal.model import stepmodel
from quetzal.io.quenedi import read_parameters, restrict_df_to_variant
from syspy.spatial.utils import get_epsg
from syspy.spatial.spatial import DBSCAN_sclustering, voronoi_diagram_dataframes, zone_clusters, get_links_hull


on_lambda = bool(os.environ.get('AWS_EXECUTION_ENV'))
num_cores = os.cpu_count()
print('num cores:', num_cores, 'on_lambda:', on_lambda)

num cores: 16 on_lambda: False


# Folders stucture and params

everything is on S3 (nothing on ECR) so no direct input folder. just scenarios/{scen}/inputs/

In [677]:
scenario = argv['scenario']
training_folder = argv['training_folder']

# if local. add the path to the scenario scenarios/<scenario>/
local_scen_path = '' if on_lambda else os.path.join('scenarios/', scenario)

input_folder = os.path.join(training_folder, 'inputs/')
scenario_folder = os.path.join(training_folder, local_scen_path, 'inputs/')
model_folder = os.path.join(training_folder, local_scen_path, 'model/')
output_folder = os.path.join(training_folder, local_scen_path, 'outputs/')

if not os.path.exists(output_folder):
    os.makedirs(output_folder)

print('input folder: ', scenario_folder)
print('output folder: ', output_folder)

input folder:  ../../scenarios/clermont_v4\inputs/
output folder:  ../../scenarios/clermont_v4\outputs/


In [678]:
params = read_parameters(argv['params'])#, period=period)

rnodes_file = os.path.join(scenario_folder, 'road/road_nodes.geojson')
rnodes_file_provided = os.path.isfile(rnodes_file)

zonage_file = os.path.join(scenario_folder, 'zonage.geojson')
zonage_file_provided = os.path.isfile(zonage_file)

od_file = os.path.join(scenario_folder, 'od/od.geojson')
od_file_provided = os.path.isfile(od_file)

# inputs

In [679]:
sm = stepmodel.read_zippedpickles(model_folder + 'cleaned')

zones: 100%|██████████| 14/14 [00:00<00:00, 43.13it/s]    


In [680]:
if sm.timetable_used:
    end_of_notebook

In [681]:
crs = sm.crs

# Folders stucture and params

everything is on S3 (nothing on ECR) so no direct input folder. just scenarios/{scen}/inputs/

In [682]:
period = argv['params'].get('pathfinder params (headway)', pathfinder_params)['period']
if 'pathfinder params (headway)' not in argv['params']:
    print(" /!\ Params json is out of date /!\ ")

if period != '':
    assert '_'+period in sm.periods, 'No headway/service hours are defined for period {} in links. Either define them in the links or choose among {}'.format(period, [per[1:] for per in sm.periods])
    sm.links['headway'] = sm.links['headway{}'.format(period)]
else:
    assert 'headway' in sm.links.columns, '"headway" column not found in links. Please select period among {}'.format(period, [per[1:] for per in sm.periods])

In [683]:
max_length = float(params['footpaths']['max_length'])
speed = float(params['footpaths']['speed'])
n_ntlegs = int(params['footpaths']['n_ntlegs'])

use_road_network = params['general'].get('use_road_network', general['use_road_network'])
plot_ODs = params['general'].get('plot_ODs', general['plot_ODs'])
n_clusters_voronoi = int(params['general'].get('n_clusters_vononoi', general['n_clusters_vononoi']))
transfer_penalty = int(params['general'].get('transfer_penalty', general['transfer_penalty']))

In [684]:
walk_on_road = (rnodes_file_provided & use_road_network)

if walk_on_road:
    sm.integrity_fix_road_nodeset_consistency()
    sm.integrity_fix_road_network()
    sm.integrity_fix_road_duplicated_ab_links()

direct: 31738it [00:00, 69821.36it/s]
reversed: 31738it [00:00, 60764.79it/s]


Reached max recursive_depth


In [685]:
print('rnodes_file_provided?', rnodes_file_provided)
print('zonage_file_provided?', zonage_file_provided)
print('od_file_provided?', od_file_provided)

print('walk_on_road?', walk_on_road)

rnodes_file_provided? True
zonage_file_provided? True
od_file_provided? True
walk_on_road? True


In [686]:
nodes_metadata = sm.nodes.copy()

In [687]:
# clustering and rename nodes
# sm.preparation_clusterize_nodes(distance_threshold=10, fast=True, prefix='cnode_')  # ~10m
# sm.nodes.index.name = 'index'

# Isochrone

In [688]:
sm_iso = sm.copy()
sm_iso.preparation_clusterize_nodes(distance_threshold=50, fast=True, prefix='cnode_')  # ~10m
sm_iso.nodes.index.name = 'index'

can not add prefixes on table:  footpaths


In [689]:
### To apply transfer penalty correctly, we need to duplicate nodes:
if transfer_penalty > 0:
    new_links = sm_iso.links.copy()
    new_nodes = sm_iso.nodes.copy()

    # Build one row per node-line association
    node_lines = pd.concat([
        new_links[['a', 'trip_id']].rename(columns={'a': 'index'}),
        new_links[['b', 'trip_id']].rename(columns={'b': 'index'})
    ], ignore_index=True).drop_duplicates().set_index('index')

    # Attach geometry from points index
    points_expanded = (
        node_lines
        .merge(new_nodes, left_index=True, right_index=True)
        .reset_index(names='index')
        .sort_values(['index', 'trip_id'])
    )

    # Offset function
    def offset_point(pt, i, n, distance=1):
        if n == 1:
            return pt
        angle = 2 * math.pi * i / n
        dx = distance * math.cos(angle)
        dy = distance * math.sin(angle)
        return Point(pt.x + dx, pt.y + dy)

    # Apply offset per original node
    def apply_offsets(group, distance=1):
        group = group.copy()
        n = len(group)
        group['geometry'] = [
            offset_point(pt, i, n, distance=distance)
            for i, pt in enumerate(group['geometry'])
        ]
        return group

    points_expanded = (
        points_expanded
        .groupby('index', group_keys=False)
        .apply(apply_offsets, distance=5)
    )

    # Create new node ids with suffix
    points_expanded['parent_index'] = points_expanded['index'].copy()
    points_expanded['index'] = (
        points_expanded['index'].astype(str) + '_' + points_expanded['trip_id'].astype(str)
    )

    # Final points dataframe
    new_nodes = (
        points_expanded
        .set_index('index')[['geometry', 'parent_index']]
        #.set_index('index')[['geometry']]
    )

    # Update links
    new_links['a'] = new_links['a'].astype(str) + '_' + new_links['trip_id'].astype(str)
    new_links['b'] = new_links['b'].astype(str) + '_' + new_links['trip_id'].astype(str)

    sm_iso.links = new_links.copy()
    sm_iso.nodes = new_nodes.copy()

In [690]:
# get envelope to clip the voronoi
clip_geometry = get_links_hull(sm_iso.links, tolerance=1000)

In [691]:
# 1) Clusterize nodes that are really close
clustered_nodes = sm_iso.nodes.copy()
label = DBSCAN_sclustering(clustered_nodes, distance_threshold=0.1)
label = 'node_cluster_' + label.astype(str)
clustered_nodes['cluster'] = label
nodes_cluster_series = clustered_nodes['cluster']

clustered_nodes = clustered_nodes.drop_duplicates('cluster')
clustered_nodes = clustered_nodes.reset_index().set_index('cluster')


In [692]:
# 2) create voronoi with nodes clusters
voronoi, _ = voronoi_diagram_dataframes(clustered_nodes['geometry'])
voronoi.crs = sm_iso.crs

voronoi = voronoi.clip(clip_geometry)

In [693]:
# 3) Clusterize the voronoi into 1000 zones
clusters, cluster_series = zone_clusters(voronoi, n_clusters=n_clusters_voronoi, geo_join_method=union_all)
clusters = gpd.GeoDataFrame(clusters)

cluster_series = 'zones_' + cluster_series.astype(str)
clusters.index = 'zones_' + clusters.index.astype(str)
clusters.index.name = 'index'

In [694]:
zone_cluster_dict = cluster_series.reset_index().groupby('cluster')['index'].agg(list).to_dict()
clusters['node_cluster_list'] = clusters.index.map(zone_cluster_dict.get)
clusters['node_cluster_id'] = clusters['node_cluster_list'].apply(lambda x: x[0])

In [695]:
node_cluster_dict = nodes_cluster_series.reset_index().groupby('cluster')['index'].agg(list).to_dict()
clusters['node_list'] = clusters['node_cluster_list'].apply(lambda x: [*map(node_cluster_dict.get, x)])
clusters['node_list'] = clusters['node_list'].apply(lambda ls: [item for sublist in ls for item in sublist])  # flatten
clusters['node_id'] = clusters['node_list'].apply(lambda x: x[0])

### pathfinder

In [696]:
zones = sm_iso.nodes.loc[clusters['node_id'].values].copy()
zones = zones.reset_index().rename(columns={'index': 'node_id'})
zone_node_dict = clusters.reset_index().set_index('node_id')['index'].to_dict()

zones.index = zones['node_id'].apply(zone_node_dict.get)
zones.index.name = 'index'

zones["geometry"] = zones['geometry'].apply(lambda x: shapely.transform(x, lambda x: x + 1)) # to avoid point-type geometries in links

In [697]:
sm_iso.zones = zones.copy()

In [698]:
if not walk_on_road:
    sm_iso.preparation_footpaths(speed=speed, max_length=max_length)
    sm_iso.footpaths = sm_iso.footpaths.drop(columns='voronoi')
    sm_iso.footpaths['speed'] = sm_iso.footpaths['length'] / sm_iso.footpaths['time'] * 3.6
    sm_iso.footpaths['duration'] = sm_iso.footpaths['time'].copy()
    sm_iso.footpaths['real_time'] = sm_iso.footpaths['time'].copy()
    sm_iso.footpaths['time'] += transfer_penalty

    if transfer_penalty > 0:
        pairs = (
            sm_iso.nodes.reset_index().copy()[["index", "parent_index"]]
            .merge(sm_iso.nodes.reset_index().copy()[["index", "parent_index"]], on="parent_index", suffixes=("_a", "_b"))
            .query("index_a != index_b")
            .rename(columns={"index_a": "a", "index_b": "b"})
            [["a", "b", "parent_index"]]
            .merge(
                sm_iso.nodes.reset_index().copy()[["index", "geometry"]].rename(columns={"index": "a", "geometry": "geom_a"}),
                on="a"
            )
            .merge(
                sm_iso.nodes.reset_index().copy()[["index", "geometry"]].rename(columns={"index": "b", "geometry": "geom_b"}),
                on="b"
            )
        )

        add_footpaths = gpd.GeoDataFrame(
            pairs[["a", "b", "parent_index"]].assign(
                geometry=[LineString([ga, gb]) for ga, gb in zip(pairs["geom_a"], pairs["geom_b"])]
            ),
            geometry="geometry",
            crs=sm.crs
        )

        add_footpaths[['length', 'time', 'speed', 'duration', 'real_time']] = (0, transfer_penalty, speed, 0, 0)

        sm_iso.footpaths = pd.concat([sm_iso.footpaths, add_footpaths]).sort_values('time').drop_duplicates(subset=['a', 'b'], keep='first')

In [699]:
# sm_iso.footpaths['parent_a'] = sm_iso.footpaths['a'].map(sm_iso.nodes['parent_index'].to_dict())
# sm_iso.footpaths['parent_b'] = sm_iso.footpaths['b'].map(sm_iso.nodes['parent_index'].to_dict())
# sm_iso.footpaths['to_mapmatch'] = (sm_iso.footpaths['parent_a'] != sm_iso.footpaths['parent_b'])
# footpaths_to_mapmatch = sm_iso.footpaths.loc[sm_iso.footpaths['to_mapmatch']]

In [700]:
# if walk_on_road and len(footpaths_to_mapmatch)>0:
#     sm_footpaths = stepmodel.StepModel(crs=sm.crs, coordinates_unit='meter')
# 
#     sm_footpaths.road_links = sm_iso.road_links.copy()
#     sm_footpaths.road_links['time'] = sm_footpaths.road_links['walk_time'].copy()
#     sm_footpaths.road_nodes = sm_iso.road_nodes.copy()
# 
#     sm_footpaths.zones = sm_iso.nodes.reset_index()[['geometry', 'index']].set_index('index')
#     od_set = {(a, b) for a, b in zip(footpaths_to_mapmatch['a'], footpaths_to_mapmatch['b'])}
# 
#     sm_footpaths.preparation_ntlegs(
#         short_leg_speed=speed,
#         long_leg_speed=speed,
#         zone_to_transit=False,
#         zone_to_road=True,
#         road_to_transit=False,
#         n_ntlegs=1
#     )
# 
#     sm_footpaths.step_road_pathfinder(
#         method='aon', 
#         od_set = od_set
#     )
#     
#     sm_footpaths.analysis_car_los()
#     sm_footpaths.analysis_car_length()
#     sm_footpaths.car_los.rename(columns={'origin': 'a', 'destination': 'b'}, inplace=True)
#     footpaths_to_mapmatch = footpaths_to_mapmatch.drop(columns=['length', 'time'])
#     
#     footpaths_to_mapmatch = pd.merge(
#         footpaths_to_mapmatch,
#         sm_footpaths.car_los[['a', 'b', 'time', 'length']].reset_index(),
#         on=['a', 'b'],
#         how='left'
#     )
# 
#     sm_iso.footpaths = pd.concat([sm_iso.footpaths.loc[~sm_iso.footpaths['to_mapmatch']], footpaths_to_mapmatch])

In [701]:
sm_iso.preparation_ntlegs(
    short_leg_speed=speed,
    long_leg_speed=speed,  # tout le monde marche
    zone_to_transit=True,
    zone_to_road=False,
    road_to_transit=walk_on_road,
    n_ntlegs=n_ntlegs,
)

sm_iso.zone_to_transit.index = 'ztt_' + sm_iso.zone_to_transit.index.astype(str)


In [702]:
if walk_on_road:
    sm_iso.road_to_transit.index = 'rtt_' + sm_iso.road_to_transit.index.astype(str)
    sm_iso.road_to_transit['length'] = sm_iso.road_to_transit['geometry'].apply(lambda x: x.length)
    sm_iso.road_to_transit['duration'] = sm_iso.road_to_transit['time'].copy()
    sm_iso.road_to_transit['real_time'] = sm_iso.road_to_transit['time'].copy()
    sm_iso.road_to_transit.loc[sm_iso.road_to_transit['direction'] == 'access', 'time'] += transfer_penalty

    sm_iso.road_links['duration'] = sm_iso.road_links['time'].copy()
    sm_iso.road_links['real_time'] = sm_iso.road_links['time'].copy()

    columns = ['a', 'b', 'geometry', 'length', 'time', 'speed', 'duration', 'real_time']
    sm_iso.footpaths = pd.concat([sm_iso.road_to_transit[columns].reset_index(), sm_iso.road_links[columns].reset_index()])
    sm_iso.footpaths.set_index('index', inplace=True)

    nodes_no_road = sm_iso.nodes.copy()
    sm_iso.nodes = pd.concat([sm_iso.nodes.reset_index(), sm_iso.road_nodes.reset_index()]).set_index('index')

connect zone to every nodes in its cluster

In [703]:
zone_to_transit = clusters[['node_list']].explode('node_list').reset_index()

zone_to_transit['rank'] = 0
zone_to_transit['distance'] = 0
zone_to_transit['geometry'] = LineString([[0, 0], [0, 0]])
zone_to_transit['direction'] = 'access'
zone_to_transit['speed_factor'] = 0
zone_to_transit['short_leg_speed'] = speed
zone_to_transit['long_leg_speed'] = speed
zone_to_transit['speed'] = speed
zone_to_transit['time'] = 0
zone_to_transit['walk_time'] = 0


In [704]:
access = zone_to_transit.rename(columns={'index': 'a', 'node_list': 'b'})

eggress = zone_to_transit.rename(columns={'index': 'b', 'node_list': 'a'})
eggress['direction'] = 'eggress'

In [705]:
sm_iso.zone_to_transit = pd.concat([sm_iso.zone_to_transit, access, eggress], ignore_index=True)  
sm_iso.zone_to_transit = sm_iso.zone_to_transit.drop_duplicates(subset=['a', 'b'], keep='first')

In [706]:
sm_iso.step_pt_pathfinder(
    broken_routes=False,
    broken_modes=False,
    keep_pathfinder=True,
    mode_column='route_type',
    route_column='route_id',
    speedup=True,
    walk_on_road=False,
    path_analysis=True,
)

start publicpathfinder
168 sources 168 targets direct search
path_analysis


path_analysis: 100%|██████████| 28224/28224 [00:01<00:00, 17313.21it/s]


In [707]:
sm_iso.footpaths['time'] = sm_iso.footpaths['real_time']

In [708]:
sm_iso.analysis_pt_time()

In [709]:
sm_iso.pt_los['time (mins)'] = sm_iso.pt_los['time']//60

In [710]:
sm_iso.nodes = nodes_no_road.copy()

In [711]:
def agg_func(x):
    return dict(x.values)

clusters.to_crs(epsg='4326')[['geometry']].to_file(os.path.join(output_folder, 'isochrone_voronoi.geojson'), driver='GeoJSON')

json_data = {}
grouped = sm_iso.pt_los.groupby(['origin', 'destination'])['time (mins)'].sum().reset_index()
grouped = grouped.loc[grouped['origin'] != grouped['destination']]
grouped['time (mins)'] = list(zip(grouped['destination'], grouped['time (mins)']))
data = grouped.groupby('origin').agg({'time (mins)': agg_func}).to_dict()
json_data.update(data)

with open(os.path.join(output_folder, 'isochrone_voronoi.json'), 'w') as json_file:
    json.dump(json_data, json_file)

In [712]:
if zonage_file_provided:
    zones_iso = sm.zones.copy()[['geometry']]

    zones_iso, zones_iso_series = zone_clusters(zones_iso, n_clusters=1000, geo_join_method=union_all)
    zones_iso = gpd.GeoDataFrame(zones_iso, crs=crs)
    zones_iso.index = 'zone_cluster_' + zones_iso.index.astype(str)
    zones_iso.index.name = 'index'
    all_zones = zones_iso.copy()
    
    zones_iso['geometry'] = zones_iso['geometry'].centroid

    nodes_iso = sm_iso.zones.copy().reset_index(names='zone_iso_id').set_index('node_id')

    zones_iso['x1'], zones_iso['y1'] = zones_iso['geometry'].apply(lambda geom: geom.x), zones_iso['geometry'].apply(lambda geom: geom.y)
    nodes_iso['x2'], nodes_iso['y2'] = nodes_iso['geometry'].apply(lambda geom: geom.x), nodes_iso['geometry'].apply(lambda geom: geom.y)

    zones_iso_pl = pl.from_pandas(zones_iso.reset_index(names='zone_id')[['zone_id', 'x1', 'y1']])
    nodes_iso_pl = pl.from_pandas(nodes_iso.reset_index(names='node_id')[['zone_iso_id', 'x2', 'y2']])


In [713]:
if zonage_file_provided:
    prep_tree = zones_iso_pl.join(nodes_iso_pl, how='cross')
    prep_tree = prep_tree.with_columns(
        np.sqrt((pl.col('x1') - pl.col('x2'))**2 + (pl.col('y1') - pl.col('y2'))**2).alias('distance')
    ).drop(['x1', 'x2', 'y1', 'y2'])

    tree_max_dist = prep_tree.filter(
        pl.col('distance') < max_length
    )
    
    if walk_on_road:
        sm_access_eggress = stepmodel.StepModel(crs=sm.crs, coordinates_unit='meter')

        sm_access_eggress.road_links = sm.road_links.copy()
        sm_access_eggress.road_links['time'] = sm_access_eggress.road_links['walk_time'].copy()
        sm_access_eggress.road_nodes = sm.road_nodes.copy()

        zones_access_eggress = zones_iso.reset_index()[['geometry', 'index']]
        nodes_access_eggress = nodes_iso.reset_index()[['geometry', 'zone_iso_id']]
        nodes_access_eggress = nodes_access_eggress.rename(columns={'zone_iso_id': 'index'})

        sm_access_eggress.zones = pd.concat([zones_access_eggress, nodes_access_eggress])
        sm_access_eggress.zones = sm_access_eggress.zones.set_index('index')
        od_set = {(zone, node) for zone, node in zip(tree_max_dist['zone_id'], tree_max_dist['zone_iso_id'])}

        sm_access_eggress.preparation_ntlegs(
            short_leg_speed=speed,
            long_leg_speed=speed,
            zone_to_transit=False,
            zone_to_road=True,
            road_to_transit=False,
            n_ntlegs=n_ntlegs
        )

        sm_access_eggress.step_road_pathfinder(
            method='aon', 
            od_set = od_set)
        
        sm_access_eggress.car_los.rename(columns={'origin': 'zone_id', 'destination': 'zone_iso_id', 'gtime': 'gtime_leg'}, inplace=True)
        
        tree_max_dist = tree_max_dist.join(
            pl.from_pandas(sm_access_eggress.car_los[['zone_id', 'zone_iso_id', 'gtime_leg']]),
            on=['zone_id', 'zone_iso_id'],
            how='left'
        )
        
    else:
        tree_max_dist = tree_max_dist.with_columns(
            (pl.col('distance')/(speed/3.6)).alias('gtime_leg')
        ).drop('distance')


    tree_max_dist_from = tree_max_dist.rename({'zone_id': 'zone_id_from', 'zone_iso_id': 'origin', 'gtime_leg': 'gtime_leg_from'})
    tree_max_dist_to = tree_max_dist.rename({'zone_id': 'zone_id_to', 'zone_iso_id': 'destination', 'gtime_leg': 'gtime_leg_to'})

    pt_los_pl = pl.from_pandas(sm_iso.pt_los[['origin', 'destination', 'gtime']])

self.volumes does not exist. od generated with self.zones, od_set


In [714]:
if zonage_file_provided:
    tree_all = pl.DataFrame()

    for zone_from in tqdm.tqdm(tree_max_dist['zone_id'].unique().to_list()):
        tree_from = tree_max_dist_from.filter(pl.col('zone_id_from') == zone_from)
        tree = pt_los_pl.join(
                tree_from, on='origin', how='right', suffix='_from'
            ).join(
                tree_max_dist_to, on='destination'
            ).with_columns(
                (pl.col('gtime') + pl.col('gtime_leg_from') + pl.col('gtime_leg_to')).alias('gtime')
            ).drop(['gtime_leg_from', 'gtime_leg_to'])

        tree = tree.group_by(
                ['zone_id_from', 'zone_id_to']
            ).agg(
                pl.all().sort_by("gtime").first()
            )
        
        tree_all = pl.concat([tree_all, tree])

100%|██████████| 26/26 [00:00<00:00, 409.73it/s]


In [715]:
if zonage_file_provided:
    tree_all_pd = tree_all.with_columns(
            (pl.col('gtime')/60).alias('time (mins)')
        ).drop(
            ['origin', 'destination', 'gtime']    
        ).rename(
            {'zone_id_from': 'origin', 'zone_id_to': 'destination'}
        ).to_pandas()

    tree_all_pd = tree_all_pd.loc[tree_all_pd['origin'] != tree_all_pd['destination']]

    json_data = {}
    tree_all_pd['time (mins)'] = list(zip(tree_all_pd['destination'], tree_all_pd['time (mins)']))
    data = tree_all_pd.groupby('origin').agg({'time (mins)': agg_func}).to_dict()
    json_data.update(data)

    with open(os.path.join(output_folder, 'isochrone_zonage.json'), 'w') as json_file:
        json.dump(json_data, json_file)

In [716]:
if zonage_file_provided:
    all_zones.loc[tree_max_dist['zone_id'].unique().to_list()].to_crs(epsg='4326')[['geometry']].to_file(os.path.join(output_folder, 'isochrone_zonage.geojson'), driver='GeoJSON')

# OD pathfinder

In [717]:
od_file = os.path.join(scenario_folder, 'od', 'od.geojson')
od_file_provided = os.path.isfile(od_file)
if od_file_provided:
    od_test = gpd.read_file(od_file)
    if 'name' not in od_test.columns:
        od_test['name'] = od_test['index']
    od_test['name'] = od_test['name'].fillna(od_test['index'].astype(str))
    od_test.to_crs(crs=sm.crs, inplace=True)
    if 'volume' in od_test.columns:
        od_test = od_test.loc[od_test['volume'] > 5]
else:
    print('end of pathfinder')
    end_of_notebook

end_of_notebook here if OD_file not provided!!

In [718]:
od_test['geometry_o'] = od_test['geometry'].apply(lambda g: Point(g.coords[:][0]))
od_test['geometry_d'] = od_test['geometry'].apply(lambda g: Point(g.coords[:][1]))

od_test['origin'] = od_test['index'].astype(str) + '_o'
od_test['destination'] = od_test['index'].astype(str) + '_d'

zones = od_test.copy()
zones_d = od_test.copy()
zones['geometry'] = zones['geometry_o']
zones_d['geometry'] = zones_d['geometry_d']
zones['index'] = zones['origin']
zones_d['index'] = zones_d['destination']
zones = pd.concat([zones[['index', 'geometry']], zones_d[['index', 'geometry']]])
zones = zones.to_crs(crs).set_index('index')

In [719]:
zones_simplify = zones.copy().reset_index()
zones_simplify = zones_simplify.groupby(['geometry']).agg({'index': list}).reset_index()
zones_simplify = zones_simplify.explode('index').reset_index(names='zone_index')
zones_simplify['zone_index'] = 'zone_' + zones_simplify['zone_index'].astype(str)
zones_simplify_dict = zones_simplify.set_index('index')['zone_index'].to_dict()
zones_simplify = zones_simplify.drop(columns=['index']).drop_duplicates().set_index('zone_index')

In [720]:
od_test[['origin_raw', 'destination_raw']] = od_test[['origin', 'destination']].copy()
od_test['origin'], od_test['destination'] = od_test['origin_raw'].map(zones_simplify_dict), od_test['destination_raw'].map(zones_simplify_dict)

sm.zones = zones_simplify.copy()

# Walkmodel

In [721]:
### To apply transfer penalty correctly, we need to duplicate nodes:
if transfer_penalty > 0:
    new_links = sm.links.copy()
    new_nodes = sm.nodes.copy()

    # Build one row per node-line association
    node_lines = pd.concat([
        new_links[['a', 'trip_id']].rename(columns={'a': 'index'}),
        new_links[['b', 'trip_id']].rename(columns={'b': 'index'})
    ], ignore_index=True).drop_duplicates().set_index('index')

    # Attach geometry from points index
    points_expanded = (
        node_lines
        .merge(new_nodes, left_index=True, right_index=True)
        .reset_index(names='index')
        .sort_values(['index', 'trip_id'])
    )

    # Offset function
    def offset_point(pt, i, n, distance=1):
        if n == 1:
            return pt
        angle = 2 * math.pi * i / n
        dx = distance * math.cos(angle)
        dy = distance * math.sin(angle)
        return Point(pt.x + dx, pt.y + dy)

    # Apply offset per original node
    def apply_offsets(group, distance=1):
        group = group.copy()
        n = len(group)
        group['geometry'] = [
            offset_point(pt, i, n, distance=distance)
            for i, pt in enumerate(group['geometry'])
        ]
        return group

    points_expanded = (
        points_expanded
        .groupby('index', group_keys=False)
        .apply(apply_offsets, distance=5)
    )

    # Create new node ids with suffix
    points_expanded['parent_index'] = points_expanded['index'].copy()
    points_expanded['index'] = (
        points_expanded['index'].astype(str) + '_' + points_expanded['trip_id'].astype(str)
    )

    # Final points dataframe
    new_nodes = (
        points_expanded
        .set_index('index')[['geometry', 'parent_index']]
    ###     .set_index('index')[['geometry']]
    )

    # Update links
    new_links['a'] = new_links['a'].astype(str) + '_' + new_links['trip_id'].astype(str)
    new_links['b'] = new_links['b'].astype(str) + '_' + new_links['trip_id'].astype(str)

    sm.links = new_links.copy()
    sm.nodes = new_nodes.copy()

In [722]:
if not walk_on_road:
    sm.preparation_footpaths(speed=speed, max_length=max_length)

    sm.footpaths = sm.footpaths.drop(columns='voronoi')
    sm.footpaths['speed'] = sm.footpaths['length'] / sm.footpaths['time'] * 3.6

    sm.footpaths['real_time'] = sm.footpaths['time'].copy()
    sm.footpaths['time'] += transfer_penalty

    if transfer_penalty > 0:
        pairs = (
            sm.nodes.reset_index().copy()[["index", "parent_index"]]
            .merge(sm.nodes.reset_index().copy()[["index", "parent_index"]], on="parent_index", suffixes=("_a", "_b"))
            .query("index_a != index_b")
            .rename(columns={"index_a": "a", "index_b": "b"})
            [["a", "b", "parent_index"]]
            .merge(
                sm.nodes.reset_index().copy()[["index", "geometry"]].rename(columns={"index": "a", "geometry": "geom_a"}),
                on="a"
            )
            .merge(
                sm.nodes.reset_index().copy()[["index", "geometry"]].rename(columns={"index": "b", "geometry": "geom_b"}),
                on="b"
            )
        )

        add_footpaths = gpd.GeoDataFrame(
            pairs[["a", "b", "parent_index"]].assign(
                geometry=[LineString([ga, gb]) for ga, gb in zip(pairs["geom_a"], pairs["geom_b"])]
            ),
            geometry="geometry",
            crs=sm.crs
        )

        add_footpaths[['length', 'time', 'speed', 'duration', 'real_time']] = (0, transfer_penalty, speed, 0, 0)

        sm.footpaths = pd.concat([sm.footpaths, add_footpaths]).sort_values('time').drop_duplicates(subset=['a', 'b'], keep='first')

In [723]:
# Access footpaths (zone_to_road and road_to_transit)
sm.preparation_ntlegs(
    short_leg_speed=speed,
    long_leg_speed=speed,  # tout le monde marche
    threshold=1000,
    zone_to_transit=(not walk_on_road),
    zone_to_road=walk_on_road,
    road_to_transit=walk_on_road,
    n_ntlegs=n_ntlegs,
    max_ntleg_length=5000
)


In [724]:
if walk_on_road:
    sm.zone_to_road.index = 'ztr_' + sm.zone_to_road.index.astype(str)
    sm.zone_to_road['real_time'] = sm.zone_to_road['time'].copy()
    
    sm.road_to_transit.index = 'rtt_' + sm.road_to_transit.index.astype(str)
    sm.road_to_transit['real_time'] = sm.road_to_transit['time'].copy()
    sm.road_to_transit.loc[sm.road_to_transit['direction']=='access', 'time'] += transfer_penalty
else:
    sm.zone_to_transit.index = 'ztt_' + sm.zone_to_transit.index.astype(str)

In [725]:
zones_simplify = zones_simplify.reset_index()

if walk_on_road:
    unconnected_zones = zones_simplify.loc[~zones_simplify['zone_index'].isin(sm.zone_to_road['a'].values.tolist())]
else:
    unconnected_zones = zones_simplify.loc[~zones_simplify['zone_index'].isin(sm.zone_to_transit['a'].values.tolist())]

unconnected_zones_list = unconnected_zones['zone_index'].values.tolist()
print('unconnected zones: ',  unconnected_zones_list)

unconnected zones:  ['zone_37', 'zone_267']


In [726]:
od_test = od_test.loc[(~od_test['origin'].isin(unconnected_zones_list)) & (~od_test['destination'].isin(unconnected_zones_list))]
od_set = set(zip(od_test['origin'], od_test['destination']))

sm.zones = sm.zones.loc[~sm.zones.index.isin(unconnected_zones_list)]

# pathfinder

In [727]:
sm.step_pt_pathfinder(
    broken_routes=False,
    broken_modes=False,
    keep_pathfinder=True,
    mode_column='route_type',
    route_column='route_id',
    speedup=True,
    walk_on_road=walk_on_road,
    path_analysis=False,
    od_set=od_set,
)

start publicpathfinder
871 sources 873 targets direct search
path_analysis


In [728]:
sm.analysis_pt_los(walk_on_road=walk_on_road)

path_analysis: 100%|██████████| 24274/24274 [00:01<00:00, 21082.02it/s]


In [729]:
if walk_on_road:
    sm.road_to_transit['time'] = sm.road_to_transit['real_time']
else:
    sm.footpaths['time'] = sm.footpaths['real_time']

# create path

In [730]:
od_route_components = od_test.merge(sm.pt_los[['origin', 'destination', 'gtime', 'link_path', 'ntlegs', 'footpaths']], on=['origin', 'destination'])

In [731]:
od_links = od_route_components.drop(columns=['ntlegs', 'footpaths']).copy()
od_links = od_links.drop(columns=['geometry', 'geometry_o', 'geometry_d', 'origin', 'destination'])
od_links = od_links.explode('link_path')

od_links = od_links.merge(sm.links[['route_color', 'geometry', 'time', 'speed']], left_on='link_path', right_index=True)
od_links['type'] = 'link'

In [732]:
od_ntlegs = od_route_components.drop(columns=['link_path', 'footpaths']).copy()
od_ntlegs = od_ntlegs.drop(columns=['geometry', 'geometry_o', 'geometry_d', 'origin', 'destination'])
od_ntlegs = od_ntlegs.explode('ntlegs')

if walk_on_road:
    ntlegs_dict = sm.zone_to_road.reset_index().set_index(['a', 'b'])['index'].to_dict()
    od_ntlegs['ntlegs'] = od_ntlegs['ntlegs'].apply(ntlegs_dict.get)
    od_ntlegs = od_ntlegs.merge(sm.zone_to_road[['geometry', 'time', 'speed']], left_on='ntlegs', right_index=True)
else:
    ntlegs_dict = sm.zone_to_transit.reset_index().set_index(['a', 'b'])['index'].to_dict()
    od_ntlegs['ntlegs'] = od_ntlegs['ntlegs'].apply(ntlegs_dict.get)
    od_ntlegs = od_ntlegs.merge(sm.zone_to_transit[['geometry', 'time', 'speed']], left_on='ntlegs', right_index=True)
    
od_ntlegs = od_ntlegs.drop(columns='ntlegs')
od_ntlegs['type'] = 'ntleg'
od_ntlegs['route_color'] = '4B4B4B'

In [733]:
od_footpaths = od_route_components.drop(columns=['ntlegs', 'link_path']).copy()

od_footpaths = od_footpaths.drop(columns=['geometry', 'geometry_o', 'geometry_d', 'origin', 'destination'])
od_footpaths = od_footpaths.explode('footpaths')
od_footpaths.dropna(subset='footpaths', inplace=True)

if walk_on_road:
    rlinks_dict = sm.road_links.reset_index().set_index(['a', 'b'])['index'].to_dict()
    od_footpaths['footpaths'] = od_footpaths['footpaths'].apply(rlinks_dict.get)
    od_footpaths = od_footpaths.merge(sm.road_links[['geometry', 'time', 'speed']], left_on='footpaths', right_index=True)

else:
    footpaths_dict = sm.footpaths.reset_index().set_index(['a', 'b'])['index'].to_dict()
    od_footpaths['footpaths'] = od_footpaths['footpaths'].apply(footpaths_dict.get)
    od_footpaths = od_footpaths.merge(sm.footpaths[['geometry', 'time', 'speed']], left_on='footpaths', right_index=True)

od_footpaths = od_footpaths.drop(columns='footpaths')
od_footpaths['type'] = 'footpaths'
od_footpaths['route_color']='666666'

In [734]:
od_route = pd.concat([od_links, od_footpaths, od_ntlegs], axis=0)

In [735]:
od_route['route_color'] = od_route['route_color'].fillna('838383').replace('null', '838383')
od_route['route_color'] = od_route['route_color'].apply(lambda c: f'#{c.strip()}')

In [736]:
od_route = od_route.rename(columns={'name': 'od_name'}).drop(columns='index')
od_route.reset_index(drop=True)
od_route.index.name = 'index'

In [737]:
od_route = gpd.GeoDataFrame(od_route, crs=sm.crs)
od_route['length'] = od_route.length

# export PT lost metrics

In [738]:
mode_list = list(sm.links['route_type'].unique())

In [739]:
sm.analysis_pt_time(walk_on_road=walk_on_road)
sm.analysis_pt_route_type(mode_list)
pt_los = od_test[['origin', 'destination', 'name']].merge(sm.pt_los, on=['origin', 'destination'])

In [740]:
pt_los['walking_time'] = pt_los['access_time'] + pt_los['footpath_time']
time_per_mode_cols = []

# get in_vehicle_time per modes
for mode in mode_list:
    time_dict = sm.links[sm.links['route_type'] == mode]['time'].to_dict()
    col = f'{mode}_time'
    pt_los[col] = pt_los['path'].apply(lambda ls: sum([time_dict.get(el, 0) for el in ls]))
    time_per_mode_cols.append(col)

In [741]:
time_cols = ['walking_time', 'waiting_time', 'boarding_time', *time_per_mode_cols]
cols = ['name', 'time', 'in_vehicle_time', *time_cols, 'ntransfers', 'route_types']
pt_los = pt_los[cols]
pt_los.index.name = 'index'

In [742]:
pt_los.to_csv(os.path.join(output_folder, 'od_los.csv'))

# Png of paths

In [743]:
from syspy.spatial.spatial import plot_points, plot_lineStrings
import numpy as np

In [744]:
import matplotlib.pyplot as plt
from PIL import Image
import requests
from io import BytesIO


def lonlat_to_meters(lon, lat):
    """Convert lon/lat to meters in Web Mercator."""
    origin_shift = 2 * np.pi * 6378137 / 2.0
    mx = lon * origin_shift / 180.0
    my = np.log(np.tan((90 + lat) * np.pi / 360.0)) * 6378137
    return mx, my


def calculate_zoom_to_fit_bbox(minx, miny, maxx, maxy, target_width_px, target_height_px, tile_size=256):
    """
    Calculate the appropriate zoom level to fit a bounding box into a fixed pixel size.
    Returns an approximate integer zoom.
    """
    # Convert bbox corners to Web Mercator meters
    mx1, my1 = lonlat_to_meters(minx, miny)
    mx2, my2 = lonlat_to_meters(maxx, maxy)

    # Get span in meters
    width_m = abs(mx2 - mx1)
    height_m = abs(my2 - my1)

    # Calculate resolution required (meters per pixel)
    res_x = width_m / target_width_px
    res_y = height_m / target_height_px
    target_res = max(res_x, res_y)

    # Initial resolution at zoom 0
    initial_res = 2 * np.pi * 6378137 / tile_size

    # Compute zoom level from resolution
    zoom = np.log2(initial_res / target_res)

    return int(round(zoom))


def deg2num(lat_deg, lon_deg, zoom):
    lat_rad = np.radians(lat_deg)
    n = 2.0**zoom
    xtile = int((lon_deg + 180.0) / 360.0 * n)
    ytile = int((1.0 - np.log(np.tan(lat_rad) + 1 / np.cos(lat_rad)) / np.pi) / 2.0 * n)
    return (xtile, ytile)


def num2deg(xtile, ytile, zoom):
    n = 2.0**zoom
    lon_deg = xtile / n * 360.0 - 180.0
    lat_rad = np.atan(np.sinh(np.pi * (1 - 2 * ytile / n)))
    lat_deg = np.degrees(lat_rad)
    return (lat_deg, lon_deg)


def create_basemap(ax, image_size_px=(512, 512)):
    miny, maxy = ax.get_ylim()
    minx, maxx = ax.get_xlim()
    # Fixed image resolution
    zoom = calculate_zoom_to_fit_bbox(minx, miny, maxx, maxy, image_size_px[0], image_size_px[1])

    print(f'Zoom : {zoom}')

    x_min, y_max = deg2num(miny, minx, zoom)
    x_max, y_min = deg2num(maxy, maxx, zoom)

    width = x_max - x_min + 1
    height = y_max - y_min + 1

    # Create a blank image
    tile_size = 256
    basemap = Image.new('RGB', (width * tile_size, height * tile_size))
    # Download and stitch tiles
    for x in range(x_min, x_max + 1):
        for y in range(y_min, y_max + 1):
            url = f'http://a.basemaps.cartocdn.com/light_nolabels/{zoom}/{x}/{y}.png'
            response = requests.get(url)
            tile = Image.open(BytesIO(response.content))
            basemap.paste(tile, ((x - x_min) * tile_size, (y - y_min) * tile_size))

    # Show basemap image
    lat_top_left, lon_top_left = num2deg(x_min, y_min, zoom)
    lat_bottom_right, lon_bottom_right = num2deg(x_max + 1, y_max + 1, zoom)
    ax.imshow(basemap, extent=[lon_top_left, lon_bottom_right, lat_bottom_right, lat_top_left], aspect='equal')
    ax.set_xlim([minx, maxx])
    ax.set_ylim([miny, maxy])


In [745]:
od_route.to_crs(epsg='4326', inplace=True)

In [746]:
if plot_ODs:
    od_route.to_file(os.path.join(output_folder, 'od_route.geojson'), driver='GeoJSON')
    

In [747]:
if plot_ODs:
    sm.zones = gpd.GeoDataFrame(sm.zones, geometry='geometry', crs='EPSG:4326')
    sm = sm.change_epsg(epsg='4326', coordinates_unit = 'degree')
    
    for origin, destination, name in od_test.head(10)[['origin', 'destination', 'name']].values:
        fig, ax = plt.subplots(figsize=(10, 10))
        route = od_route[od_route['od_name'] == name]

        route_stations = route.loc[route['type'] == 'link'].copy()
        route_stations = route_stations[['route_color', 'geometry']]
        route_stations['node_from'] = route_stations['geometry'].apply(lambda x: Point(x.coords[0]))
        route_stations['node_to'] = route_stations['geometry'].apply(lambda x: Point(x.coords[-1]))
        route_stations_all = pd.concat([route_stations[['route_color', 'node_from']].rename(columns={'node_from': 'geometry'}), route_stations[['route_color', 'node_to']].rename(columns={'node_to': 'geometry'})])
        route_stations_all = route_stations_all.drop_duplicates()

        sm.od_basemap(origin=origin, destination=destination, squared=True, figsize=(10, 10), ax=ax)
        if len(route_stations_all) > 0:
            plot_points(route_stations_all, ax=ax, s=15, color=route_stations_all['route_color'])
        plot_lineStrings(route, ax=ax, linewidth=2, colors=route['route_color'])
        create_basemap(ax)

        sm.zones.loc[[origin]].plot(ax=ax, color='green')
        sm.zones.loc[[destination]].plot(ax=ax, color='red')

        ax.set_xticks([])
        ax.set_yticks([])

        title = f'{name} \n'

        path = pt_los[pt_los['name'] == name].iloc[0]
        path['min'] = np.round(path['time'] / 60).astype(int)
        mins = (
            (path[['in_vehicle_time', 'walking_time', 'waiting_time', 'boarding_time', 'time']] / 60)
            .astype(int)
            .astype(str)
        )
        
        walking_links = route.loc[route['type']=='footpaths']
        if len(walking_links) >= 1:
            walking_distance = f"' ({int(walking_links['length'].sum()/1000)} kms)"
        else:
            walking_distance = "'"

        title += f'{path['ntransfers']} transfers | {path['min']} mins '
        title += '\n' + 'in vehicle ' + mins['in_vehicle_time'] + "' | " + ' waiting ' + mins['waiting_time'] + "' | "
        title += 'walking ' + mins['walking_time'] + walking_distance + " | " + ' boarding ' + mins['boarding_time'] + "'" + '\n'

        trip_dict = sm.links['trip_id'].to_dict()
        trip_list = route['link_path'].apply(trip_dict.get).drop_duplicates().dropna().to_list()
        title += '->'.join(trip_list)
        ax.set_title(title)

        png = f'OD_PT_{name}.png'
        fig.savefig(os.path.join(output_folder, png), bbox_inches='tight')

In [748]:
end_of_notebook

NameError: name 'end_of_notebook' is not defined

# Bar plot

In [ ]:
import matplotlib.pyplot as plt

systra_colors = [
    '#74a9cf',
    '#0570b0',
    '#bdc9e1',
    '#b2df8a',
    '#33a02c',
    '#fb9a99',
    '#e31a1c',
    '#fdbf6f',
    '#ff7f00',
    '#cab2d6',
    '#6a3d9a',
]


In [ ]:
to_plot = pt_los.copy()
to_plot = to_plot.set_index('name')
to_plot[time_cols] = to_plot[time_cols] / 60

In [ ]:
# ax = to_plot[time_cols].plot(kind='barh', stacked=True, figsize=(10, 10), color=systra_colors)
# plt.legend(loc='upper right', ncol=1)
# plt.gca().invert_yaxis()
# plt.xlabel('Time (mins)')
# plt.grid(True, 'major', linestyle='-', axis='both')
# ax.set_axisbelow(True)
# plt.legend(time_cols, loc='center left', bbox_to_anchor=(1, 0.5), ncol=1)
# plt.title('Time decomposition for each OD best path')
# plt.savefig(os.path.join(output_folder, 'od_time_decomposition.png'))